# FINAL — Traffic Sign Detection only

For the current `video1.mp4`: detect Vietnamese traffic signs only, draw short English labels, save the full result to Google Drive, and show a lightweight preview in Colab.

Before **Runtime → Run all**, select **T4 GPU**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/DIP
!git clone -q -b feature/yolo-traffic-safety https://github.com/NVTruong473/DIP.git /content/DIP
%cd /content/DIP/END_DIP
!pip install -q -r requirements.txt


In [ ]:
from pathlib import Path
import os, subprocess, torch

ROOT = Path('/content/drive/MyDrive/DIP')
VIDEO_NAME = 'video1.mp4'
VIDEO = ROOT / VIDEO_NAME
MODELS = ROOT / 'models'
OUTPUTS = ROOT / 'outputs'
MODELS.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'Enable T4 GPU: Runtime → Change runtime type → T4 GPU'
assert VIDEO.exists(), f'Missing input video: {VIDEO}'
print('GPU:', torch.cuda.get_device_name(0))
print('Input:', VIDEO)


In [ ]:
# Download once; later runs reuse the same model from MyDrive/DIP/models/traffic_sign/
subprocess.run(['python','download_models.py','--models-dir',str(MODELS)], check=True)


In [ ]:
# Remove stale output, then run traffic-sign detection only.
OUT = OUTPUTS / f'{VIDEO.stem}_result.mp4'
CSV = OUTPUTS / f'{VIDEO.stem}_result.csv'
TEMP = OUTPUTS / f'{VIDEO.stem}_result_temp.mp4'
for p in (OUT, CSV, TEMP):
    try: p.unlink()
    except FileNotFoundError: pass

cmd = ['python','main.py','--input',str(VIDEO),'--output-dir',str(OUTPUTS),'--models-dir',str(MODELS),'--sign-conf','0.25','--sign-imgsz','640','--frame-stride','1']
subprocess.run(cmd, check=True)
assert OUT.exists() and OUT.stat().st_size > 0, f'Output was not created: {OUT}'
print('Saved full result:', OUT)
print('CSV:', CSV)


In [ ]:
# Browser-friendly preview directly below this cell.
from IPython.display import Video, display
PREVIEW = Path('/content') / f'{VIDEO.stem}_result_preview.mp4'
try: PREVIEW.unlink()
except FileNotFoundError: pass
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(OUT),'-vf','scale=960:-2','-c:v','libx264','-preset','veryfast','-crf','29','-pix_fmt','yuv420p','-tag:v','avc1','-movflags','+faststart','-an',str(PREVIEW)], check=True)
display(Video(str(PREVIEW), embed=True, width=960, html_attributes='controls'))


## Recovery after Colab disconnect
The model and final result live in Google Drive. If the runtime is lost, run only the cell below; it restores the code, reuses the cached traffic-sign model, reruns `video1.mp4`, and shows the preview. No training is involved.


In [ ]:
# SINGLE RECOVERY / RERUN CELL
import os, sys, subprocess
from pathlib import Path
if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')
repo = Path('/content/DIP')
if not repo.exists():
    subprocess.run(['git','clone','-q','-b','feature/yolo-traffic-safety','https://github.com/NVTruong473/DIP.git',str(repo)], check=True)
work = repo / 'END_DIP'
os.chdir(work)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)
root = Path('/content/drive/MyDrive/DIP')
video = root / 'video1.mp4'
models = root / 'models'
outputs = root / 'outputs'
models.mkdir(parents=True, exist_ok=True); outputs.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable,'download_models.py','--models-dir',str(models)], check=True)
subprocess.run([sys.executable,'main.py','--input',str(video),'--output-dir',str(outputs),'--models-dir',str(models),'--sign-conf','0.25','--sign-imgsz','640'], check=True)
out = outputs / 'video1_result.mp4'
preview = Path('/content/video1_result_preview.mp4')
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(out),'-vf','scale=960:-2','-c:v','libx264','-crf','29','-pix_fmt','yuv420p','-an',str(preview)], check=True)
from IPython.display import Video, display
print('Reused model from:', models / 'traffic_sign')
print('Saved:', out)
display(Video(str(preview), embed=True, width=960, html_attributes='controls'))
